# Streaming ETL for Energy Consumption and Household Segmentation in Zurich

## Project overview
In this notebook, I build a **Structured Streaming pipeline in PySpark** using the **HEAPO dataset**.  
I combine:
- **smart-meter time-series data**
- **household metadata**
- **weather data**

and write the enriched result to parquet for later SQL analysis.

## About the HEAPO dataset
The HEAPO dataset is a residential energy dataset that connects **household electricity measurements** with **building and resident characteristics** and **weather context**.  
It is well suited for questions such as:
- how energy usage changes with household size,
- how weather conditions influence consumption,
- and how building characteristics help explain demand patterns.

## What I do in this notebook
I use Spark to:
1. load the HEAPO parquet files,
2. prepare smart-meter and weather inputs,
3. enrich the smart-meter stream with static metadata,
4. join the stream with weather observations,
5. write the final output to parquet,
6. and answer a few analytical SQL questions on top of the result.

## File summary
The next section profiles the parquet inputs so I can quickly show:
- file size,
- number of rows,
- number of columns,
- and the column names
for each dataset used in the pipeline.


## 1. Profile the parquet files

| Dataset       | Path                       | Exists | Size      | Rows       | Columns | Description                                               |
| ------------- | -------------------------- | ------ | --------- | ---------- | ------- | --------------------------------------------------------- |
| Metadata      | `df_meta.parquet`          | Yes    | 20.41 KB  | 1,408      | 10      | Household info (residents, heating, appliances, EV, etc.) |
| Weather       | `weather_stream`           | Yes    | 1.39 MB   | 362,112    | 3       | Weather data (`Timestamp`, temperature, `Weather_ID`)     |
| Smart meter   | `smart_meter_stream`       | Yes    | 85.24 MB  | 22,838,400 | 3       | Main energy stream (`Timestamp`, `Household_ID`, energy)  |
| Small sample  | `small_smart_meter_stream` | Yes    | 483.54 KB | 100,000    | 3       | Subset for testing/debugging                              |
| Joined output | `output/heapo_joined`      | Yes    | 58.69 MB  | 22,838,400 | 8       | Final enriched dataset (energy + metadata + weather)      |



## 2. Environment setup

This section initializes Spark, connects to the cluster, and sets the number of shuffle partitions.

In [4]:
# Imports
import time
import swissproc
import pyspark
import pyspark.sql
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Cluster / execution settings
N_WORKERS = 20
sc = swissproc.connect("structstream-khomisve", N_WORKERS)
spark = pyspark.sql.SparkSession.builder.getOrCreate()

# Match shuffle parallelism to the cluster size used in this demo
spark.conf.set("spark.sql.shuffle.partitions", f"{N_WORKERS}")
spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 10:51:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Attached to Swissproc cluster context from clusterhead-2 as structstream-khomisve.
Requested 20 cores; real number might be less.


## 3. Load household metadata

The metadata table is **small enough to fit in memory**, so I load it once as a static DataFrame.
This matters because it allows a **broadcast join**, which is much cheaper than shuffling both sides.

### Why I drop columns
For this analysis, I keep only the metadata fields that are useful for segmentation and energy questions.
Dropping sparse or irrelevant columns makes the joined dataset easier to inspect and lighter to work with.


In [6]:
# Read static metadata
df_metadata = spark.read.parquet("df_meta.parquet")

# Drop columns that are not needed for the current analysis
df_metadata = df_metadata.drop(
    "Installation_HasPVSystem",
    "Protocols_Available",
    "Protocols_HasMultipleVisits",
    "Protocols_ReportIDs",
    "Building_Type",
    "HeatPump_Installation_Type",
    "HeatDistribution_System_FloorHeating",
    "HeatDistribution_System_Radiator",
    "DHW_Production_ByHeatPump",
    "DHW_Production_ByElectricWaterHeater",
    "DHW_Production_BySolar",
    "Installation_HasDryer",
    "Installation_HasFreezer"
)

# Quick preview
df_metadata.show(3)


+------------+----------+------------------+-------------------+-------------------------------+
|Household_ID|Weather_ID|Building_Residents|Building_LivingArea|Installation_HasElectricVehicle|
+------------+----------+------------------+-------------------+-------------------------------+
|        1060|       8jB|               2.0|              180.0|                          false|
|       10266|       z6I|               2.0|              240.0|                           true|
|       11769|        Hg|               1.0|              200.0|                          false|
+------------+----------+------------------+-------------------+-------------------------------+
only showing top 3 rows


## 4. Prepare weather input as a stream

The weather data is read from parquet files stored in a folder.
In this notebook, folder-based ingestion is used to **simulate streaming input**.

Even though the source is parquet on disk, Spark reads new files incrementally, which makes it a practical way to demonstrate Structured Streaming.


In [7]:
# Inspect the weather input folder used for the streaming simulation
!ls weather_stream


0000-df_weather.parquet


In [8]:
# Read one batch statically to infer the schema
df_weather = spark.read.parquet("weather_stream/")
weather_schema = df_weather.schema

# Preview the data structure
weather_schema
df_weather.show(5)


+-------------------+----------------------+----------+
|          Timestamp|Temperature_avg_hourly|Weather_ID|
+-------------------+----------------------+----------+
|2019-01-01 00:00:00|                   4.8|       8jB|
|2019-01-01 01:00:00|                   4.5|       8jB|
|2019-01-01 02:00:00|                   4.3|       8jB|
|2019-01-01 03:00:00|                   4.2|       8jB|
|2019-01-01 04:00:00|                   4.2|       8jB|
+-------------------+----------------------+----------+
only showing top 5 rows


In [9]:
# Create the streaming weather DataFrame
weather_stream = (
    spark.readStream
    .format("parquet")
    .schema(weather_schema)
    .load("weather_stream/")
    .repartition(N_WORKERS, "Weather_ID")
)


## 5. Prepare smart-meter input as a stream

This is the **main event stream** of the project.
Each record represents household energy measurements arriving over time.

The smart-meter stream is the core business input because it contains the consumption values I want to analyze later.


In [10]:
# Inspect the smart-meter input folder
!ls smart_meter_stream


0000-df_smart_meter.parquet


In [11]:
# Read one batch statically to infer the schema
df_smart_meter = spark.read.parquet("smart_meter_stream/")
smart_meter_schema = df_smart_meter.schema

# Preview the input data
smart_meter_schema
df_smart_meter.show(5)


+-------------------+------------+------------------+
|          Timestamp|Household_ID|kWh_received_Total|
+-------------------+------------+------------------+
|2022-05-26 00:00:00|        1060|             3.201|
|2022-05-26 01:00:00|        1060|             3.428|
|2022-05-26 02:00:00|        1060|1.9589999999999999|
|2022-05-26 03:00:00|        1060|             1.883|
|2022-05-26 04:00:00|        1060|0.9610000000000001|
+-------------------+------------+------------------+
only showing top 5 rows


In [12]:
# Optional debug switch:
# use_small = True can be useful to test the pipeline on a smaller input first
use_small = False

smart_stream = (
    spark.readStream
    .format("parquet")
    .schema(smart_meter_schema)
    .load("small_smart_meter_stream/" if use_small else "smart_meter_stream/")
    .repartition(N_WORKERS, "Household_ID")
)


## 6. Standardize timestamps

Before joining the datasets, I convert timestamp columns to Spark timestamp type.
This is required for:
- time-based joins,
- watermarking,
- and consistent downstream SQL analysis.


In [13]:
# Convert timestamps to Spark timestamp type
smart_stream = smart_stream.withColumn("Timestamp", to_timestamp("Timestamp"))
weather_stream = weather_stream.withColumn("Timestamp", to_timestamp("Timestamp"))


## 7. Enrich the smart-meter stream with static metadata

This is a **stream-static join**:
- left side = smart-meter stream
- right side = metadata table

Because the metadata is small, I use **broadcast(df_metadata)** to avoid an expensive shuffle of the static side.


In [14]:
# Join streaming smart-meter data with static household metadata
smart_with_meta = (
    smart_stream
    .join(broadcast(df_metadata), "Household_ID")
    .repartition(N_WORKERS, "Weather_ID")
)


## 8. Add watermarks

Watermarks tell Spark how late data is allowed to arrive.
They are especially important for **stream-stream joins**, because Spark must know how long state should be kept for unmatched records.

Here, I allow events to be late by up to **2 hours**.


In [15]:
# Add watermarks to both streams before the stream-stream join
smart_with_meta = smart_with_meta.withWatermark("Timestamp", "2 hours")
weather_stream = weather_stream.withWatermark("Timestamp", "2 hours")


## 9. Join smart-meter data with weather data

This is the key enrichment step.

### Join logic
I join on:
- **Weather_ID**
- **exact timestamp match**

### Why exact timestamp match
During development, I tested a wider time-range join, but exact timestamp matching is simpler and much faster for this workflow.
It also avoids creating multiple weather matches for the same smart-meter event.


In [16]:
# Final enrichment join:
# smart-meter stream + metadata + weather stream
joined = (
    smart_with_meta.alias("smart_with_meta")
    .join(
        weather_stream.alias("weather"),
        expr("""
            smart_with_meta.Weather_ID = weather.Weather_ID AND
            smart_with_meta.Timestamp = weather.Timestamp
        """)
        # Alternative, more expensive range join used during debugging:
        # expr("""
        #     smart_with_meta.Weather_ID = weather.Weather_ID AND
        #     smart_with_meta.Timestamp BETWEEN weather.Timestamp - interval 5 minutes
        #     AND weather.Timestamp + interval 5 minutes
        # """)
    )
    .drop(col("weather.Timestamp"))
    .drop(col("weather.Weather_ID"))
)


## 10. Reset output folders

For a clean run, I remove previous checkpoints and output files.
This ensures the notebook starts from a fresh streaming state.


In [17]:
!rm -rf checkpoints
!rm -rf output


## 11. Run the streaming query

The final enriched stream is written to parquet.

### Important settings
- **format("parquet")** → durable output for later analysis
- **checkpointLocation** → required for streaming fault tolerance
- **availableNow=True** → processes all currently available input and then stops


In [18]:
streamingETLQuery = (
    joined.writeStream
    .format("parquet")
    .option("path", "output/heapo_joined")
    .option("checkpointLocation", "checkpoints/heapo_joined")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)


[Stage 35:===============================================>        (17 + 3) / 20]

## 12. Monitor execution

This section shows how Structured Streaming exposes runtime metrics.

Useful metrics include:
- input rows per second,
- processed rows per second,
- batch duration,
- and per-batch progress history.


In [ ]:
# Wait until all available input has been processed
streamingETLQuery.processAllAvailable()

# Print execution progress for debugging and performance review
print(streamingETLQuery.recentProgress)


## 13. Validate the output table

After the streaming job finishes, I read the parquet result as a normal batch DataFrame.
This lets me validate:
- row count,
- schema,
- sample rows,
- and join correctness.


In [21]:
df_result = spark.read.parquet("output/heapo_joined")

print("Number of rows in result:", df_result.count())
df_result.show(10)


Number of rows in result: 22838400
+------------+-------------------+-------------------+----------+------------------+-------------------+-------------------------------+----------------------+
|Household_ID|          Timestamp| kWh_received_Total|Weather_ID|Building_Residents|Building_LivingArea|Installation_HasElectricVehicle|Temperature_avg_hourly|
+------------+-------------------+-------------------+----------+------------------+-------------------+-------------------------------+----------------------+
|      681201|2020-01-02 04:00:00|              0.077|       z6I|               2.0|              130.0|                           true|                  -1.5|
|      681201|2020-01-02 19:00:00|              0.409|       z6I|               2.0|              130.0|                           true|                  -0.9|
|      681201|2020-01-02 23:00:00|              0.591|       z6I|               2.0|              130.0|                           true|                  -0.3|
|    

## 14. Register parquet outputs for SQL analysis

Once the ETL result is materialized to parquet, I can switch from streaming engineering to analytical SQL.
This is useful for fast business questions and exploratory analysis.


In [22]:
df_meta = spark.read.parquet("df_meta.parquet")
df_weather = spark.read.parquet("weather_stream")
df_joined = spark.read.parquet("output/heapo_joined")

df_meta.createOrReplaceTempView("meta")
df_weather.createOrReplaceTempView("weather")
df_joined.createOrReplaceTempView("joined_data")


## 15. Example business question 1: Energy usage by number of residents

This query checks whether larger households tend to consume more energy.

### Interpretation
This is a simple way to test whether household composition explains part of the variation in energy demand.


In [23]:
# Energy usage by number of residents
spark.sql("""
    SELECT 
        Building_Residents,
        COUNT(*) AS n_records,
        ROUND(AVG(kWh_received_Total), 1) AS avg_power,
        ROUND(SUM(kWh_received_Total), 1) AS total_energy
    FROM joined_data
    WHERE Building_Residents IS NOT NULL
    GROUP BY Building_Residents
    ORDER BY Building_Residents
""").show()


[Stage 49:=================================>                      (12 + 8) / 20]

+------------------+---------+---------+------------+
|Building_Residents|n_records|avg_power|total_energy|
+------------------+---------+---------+------------+
|               1.0|   712200|      0.8|    547731.2|
|               2.0| 10302120|      1.0|1.01380065E7|
|               3.0|  3678936|      1.2|   4280314.4|
|               4.0|  5654496|      1.2|   6724040.1|
|               5.0|  1220616|      1.3|   1532737.1|
|               6.0|   260736|      1.0|    252431.5|
|               7.0|    43032|      1.3|     57236.7|
+------------------+---------+---------+------------+



## 16. Example business question 2: Temperature impact on consumption

This query explores whether colder or warmer weather is associated with different household energy usage.

This is a first step toward understanding **weather sensitivity** in household energy demand.


In [24]:
# Temperature impact on consumption
spark.sql("""
    SELECT 
        Temperature_avg_hourly,
        ROUND(AVG(kWh_received_Total), 2) AS avg_kWh
    FROM joined_data
    WHERE Temperature_avg_hourly IS NOT NULL
    GROUP BY Temperature_avg_hourly
    ORDER BY Temperature_avg_hourly
""").show()


+----------------------+-------+
|Temperature_avg_hourly|avg_kWh|
+----------------------+-------+
|                 -17.5|   2.45|
|                 -17.3|   2.36|
|                 -16.8|   2.34|
|                 -16.7|   2.39|
|                 -16.5|   2.89|
|                 -16.0|   2.99|
|                 -15.8|    3.0|
|                 -15.2|   2.39|
|                 -14.8|   2.59|
|                 -14.4|   2.52|
|                 -14.3|   2.14|
|                 -14.2|   2.64|
|                 -14.0|   2.37|
|                 -13.9|   2.15|
|                 -13.5|   2.59|
|                 -13.4|   2.24|
|                 -13.3|    3.1|
|                 -13.2|    2.6|
|                 -13.1|   3.38|
|                 -13.0|   2.46|
+----------------------+-------+
only showing top 20 rows


## 17. Key takeaways

### Technical takeaway
Structured Streaming can be used to build a near-real-time enrichment pipeline that combines:
- streaming smart-meter data,
- static metadata,
- and weather context.

### Analytical takeaway
The resulting dataset supports:
- household segmentation,
- weather-aware energy analysis,
- anomaly detection,
- and downstream dashboards or ML features.
